In [3]:
import numpy as np
import matplotlib.pyplot as plt
from sklearn.model_selection import TimeSeriesSplit
from cnn_model_yin import CNN, cross_validate, train
import torch
import torch.nn as nn
import random
import os, sys

# load data through the data preprocessor
sys.path.append(os.path.abspath('..'))  # add parent directory to sys.path
from data_cleanup import DataProcessor

# Reproducibility (best-effort)
SEED = 42
np.random.seed(SEED)
random.seed(SEED)
torch.manual_seed(SEED)

In [4]:
INPUT_STEPS = 24
OUTPUT_STEPS = 24

processor = DataProcessor(input_steps=INPUT_STEPS, output_steps=OUTPUT_STEPS)
Train, Val, Test = processor.load_and_process_data()

X_train, y_train = Train
X_val, y_val = Val
X_test, y_test = Test

def make_model():
    return CNN(8, INPUT_STEPS, OUTPUT_STEPS, kernel_size=3, pool_kernel=0, padding=False) # input all features, no pooling, no padding

Step 1/5: Fetching, cleaning, and engineering features...


/Users/felipejaracaceres/Documents/UCD/Machine Learning/ECS171G13/venv/lib/python3.11/site-packages/ucimlrepo/fetch.py:97: DtypeWarning: Columns (2,3,4,5,6,7) have mixed types. Specify dtype option on import or set low_memory=False.
  df = pd.read_csv(data_url)
/Users/felipejaracaceres/Documents/UCD/Machine Learning/ECS171G13/data_cleanup.py:135: FutureWarning: DataFrame.fillna with 'method' is deprecated and will raise in a future version. Use obj.ffill() or obj.bfill() instead.
  df = df.fillna(method='ffill')


Step 2/5: Resampling data to hourly and setting 'Global_active_power' as target...


/Users/felipejaracaceres/Documents/UCD/Machine Learning/ECS171G13/data_cleanup.py:175: FutureWarning: 'H' is deprecated and will be removed in a future version, please use 'h' instead.
  df_hourly = df.resample('H').agg(agg_dict)


Step 3/5: Splitting data and applying scaler...
Step 4/5: Creating time-series windows...
Step 5/5: Data processing complete.


/Users/felipejaracaceres/Documents/UCD/Machine Learning/ECS171G13/data_cleanup.py:176: FutureWarning: DataFrame.fillna with 'method' is deprecated and will raise in a future version. Use obj.ffill() or obj.bfill() instead.
  df_hourly = df_hourly.fillna(method='ffill')


In [8]:
# Run cross-validation quickly (small epochs for demo) WITH checkpoint saving

n_splits = 5
tscv = TimeSeriesSplit(n_splits=n_splits)
folds = []
for train_idx, val_idx in tscv.split(X_train):
    Xtr = X_train[train_idx]
    ytr = y_train[train_idx]
    Xval = X_train[val_idx]
    yval = y_train[val_idx]
    folds.append(((Xtr, ytr), (Xval, yval)))

print(f'Constructed {len(folds)} folds. Example fold shapes:')
print('fold0 train X', folds[0][0][0].shape, 'y', folds[0][0][1].shape, 'val X', folds[0][1][0].shape)

histories, val_losses, best_cv_checkpoint_path = cross_validate(
    make_model,
    folds,
    device=None,
    epochs=10,
    batch_size=32,
    lr=1e-3,
    verbose=True,
    checkpoint_dir='checkpoint/cv_24to24',
    save_best_only=True
)
print('Per-fold final val losses:', val_losses)
print('Best CV checkpoint path:', best_cv_checkpoint_path)
print('Best per-fold checkpoint directories saved under: checkpoint/cv_24to24/')

Constructed 5 folds. Example fold shapes:
fold0 train X (4315, 24, 8) y (4315, 24) val X (4313, 24, 8)
Starting fold 1/5: train (4315, 24, 8) | val (4313, 24, 8)
Epoch 1/10 - train_loss: 0.027306 - val_loss: 0.017526
Epoch 2/10 - train_loss: 0.023887 - val_loss: 0.016698
Epoch 3/10 - train_loss: 0.022037 - val_loss: 0.016318
Epoch 4/10 - train_loss: 0.020968 - val_loss: 0.015727
Epoch 5/10 - train_loss: 0.020126 - val_loss: 0.015436
Epoch 6/10 - train_loss: 0.019552 - val_loss: 0.015259
Epoch 7/10 - train_loss: 0.019085 - val_loss: 0.015133
Epoch 8/10 - train_loss: 0.018672 - val_loss: 0.015087
Epoch 9/10 - train_loss: 0.018194 - val_loss: 0.015280
Epoch 10/10 - train_loss: 0.018044 - val_loss: 0.015117
Starting fold 2/5: train (8628, 24, 8) | val (4313, 24, 8)
Epoch 1/10 - train_loss: 0.021142 - val_loss: 0.018360
Epoch 2/10 - train_loss: 0.018405 - val_loss: 0.017664
Epoch 3/10 - train_loss: 0.017036 - val_loss: 0.017079
Epoch 4/10 - train_loss: 0.016285 - val_loss: 0.016943
Epoch 5/